# Exploratory Data Analysis — BTC-USD Sentiment-Enhanced Forecasting

This notebook explores the model-ready dataset that merges:
- **Quantitative features**: OHLCV, technical indicators (SMA, momentum, volatility)
- **Sentiment features**: Fear & Greed Index, news headline sentiment (FinBERT)

**Target**: Next-day price direction (1 = Up, 0 = Down)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="darkgrid", palette="muted")
plt.rcParams["figure.figsize"] = (14, 6)
plt.rcParams["figure.dpi"] = 100

DATA_DIR = Path("../data/processed")
df = pd.read_parquet(DATA_DIR / "btc_usd_model_ready.parquet")
print(f"Dataset: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df.index.min().date()} → {df.index.max().date()}")
df.head(3)

## 1. Dataset Overview

In [ ]:
print("Missing values per column:")
print(df.isnull().sum()[df.isnull().sum() > 0].to_string() or "  None")
print(f"\nData types:\n{df.dtypes.value_counts().to_string()}")
print(f"\nDescriptive stats (key features):")
df[["close", "daily_return", "volatility_20d", "fng_value", "fng_normalized", "target"]].describe().round(4)

## 2. Target Distribution

A balanced target is important for classification — heavy imbalance would bias accuracy metrics.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

counts = df["target"].value_counts()
labels = ["Down (0)", "Up (1)"]
colors = ["#e74c3c", "#2ecc71"]

axes[0].bar(labels, counts.sort_index().values, color=colors, edgecolor="white")
axes[0].set_title("Target Distribution (Count)")
axes[0].set_ylabel("Days")
for i, v in enumerate(counts.sort_index().values):
    axes[0].text(i, v + 5, str(v), ha="center", fontweight="bold")

axes[1].pie(counts.sort_index().values, labels=labels, colors=colors,
            autopct="%1.1f%%", startangle=90, textprops={"fontsize": 12})
axes[1].set_title("Target Distribution (%)")

plt.suptitle("Next-Day Price Direction", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 3. Price vs. Fear & Greed Index

The core hypothesis: sentiment signals (Fear & Greed) correlate with — and potentially lead — price movements.

In [ ]:
fig, ax1 = plt.subplots(figsize=(14, 6))

color_price = "#2c3e50"
color_fng = "#e67e22"

ax1.plot(df.index, df["close"], color=color_price, linewidth=1.2, label="BTC Close Price")
ax1.set_ylabel("BTC Price (USD)", color=color_price, fontsize=12)
ax1.tick_params(axis="y", labelcolor=color_price)
ax1.set_xlabel("Date")

ax2 = ax1.twinx()
ax2.fill_between(df.index, df["fng_value"], alpha=0.3, color=color_fng, label="Fear & Greed")
ax2.axhline(y=20, color="red", linestyle="--", alpha=0.5, linewidth=0.8)
ax2.axhline(y=80, color="green", linestyle="--", alpha=0.5, linewidth=0.8)
ax2.set_ylabel("Fear & Greed Index (0-100)", color=color_fng, fontsize=12)
ax2.tick_params(axis="y", labelcolor=color_fng)
ax2.set_ylim(0, 100)

ax2.text(df.index[-1], 15, "Extreme Fear", color="red", fontsize=9, ha="right", alpha=0.7)
ax2.text(df.index[-1], 85, "Extreme Greed", color="green", fontsize=9, ha="right", alpha=0.7)

fig.suptitle("BTC Price vs. Crypto Fear & Greed Index", fontsize=14, fontweight="bold")
fig.legend(loc="upper left", bbox_to_anchor=(0.1, 0.92))
plt.tight_layout()
plt.show()

## 4. Feature Distributions by Target Class

Do features separate the two classes? Overlapping distributions = weak predictors; separated = strong.

In [ ]:
key_features = [
    "daily_return", "momentum_5d", "volatility_20d",
    "fng_normalized", "fng_sma_7", "fng_momentum_5d",
    "price_sentiment_divergence", "sma_crossover", "extreme_fear",
]

fig, axes = plt.subplots(3, 3, figsize=(15, 12))
axes = axes.flatten()

for i, feat in enumerate(key_features):
    ax = axes[i]
    for label, color, name in [(0, "#e74c3c", "Down"), (1, "#2ecc71", "Up")]:
        subset = df[df["target"] == label][feat].dropna()
        ax.hist(subset, bins=40, alpha=0.5, color=color, label=name, density=True)
    ax.set_title(feat, fontsize=11)
    ax.legend(fontsize=9)

plt.suptitle("Feature Distributions by Target Class", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 5. Correlation Heatmap

Identifies multicollinearity (highly correlated features to potentially drop) and features most correlated with the target.

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
corr = df[numeric_cols].corr()

fig, ax = plt.subplots(figsize=(18, 15))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap="RdBu_r", center=0,
            vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
ax.set_title("Feature Correlation Matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
target_corr = corr["target"].drop("target").sort_values(key=abs, ascending=False)
print("Top 15 features correlated with target (next-day direction):\n")
for feat, val in target_corr.head(15).items():
    bar = "█" * int(abs(val) * 50)
    sign = "+" if val > 0 else "-"
    print(f"  {feat:<35s} {sign}{abs(val):.4f}  {bar}")

## 6. Sentiment Regime Analysis

How does BTC perform under different Fear & Greed regimes? This directly tests whether sentiment has predictive value.

In [ ]:
bins = [0, 20, 40, 60, 80, 100]
labels_fng = ["Extreme Fear", "Fear", "Neutral", "Greed", "Extreme Greed"]
df["fng_regime"] = pd.cut(df["fng_value"], bins=bins, labels=labels_fng, include_lowest=True)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Next-day up probability by regime
regime_stats = df.groupby("fng_regime", observed=True).agg(
    up_pct=("target", "mean"),
    avg_return=("daily_return", "mean"),
    count=("target", "count"),
).round(4)

colors_regime = ["#c0392b", "#e74c3c", "#95a5a6", "#2ecc71", "#27ae60"]

axes[0].bar(regime_stats.index, regime_stats["up_pct"] * 100, color=colors_regime, edgecolor="white")
axes[0].axhline(y=50, color="gray", linestyle="--", alpha=0.6)
axes[0].set_ylabel("Next-Day Up %")
axes[0].set_title("P(Up) by Sentiment Regime")
axes[0].tick_params(axis="x", rotation=30)

axes[1].bar(regime_stats.index, regime_stats["avg_return"] * 100, color=colors_regime, edgecolor="white")
axes[1].axhline(y=0, color="gray", linestyle="--", alpha=0.6)
axes[1].set_ylabel("Avg Daily Return %")
axes[1].set_title("Avg Return by Sentiment Regime")
axes[1].tick_params(axis="x", rotation=30)

axes[2].bar(regime_stats.index, regime_stats["count"], color=colors_regime, edgecolor="white")
axes[2].set_ylabel("Number of Days")
axes[2].set_title("Days per Regime")
axes[2].tick_params(axis="x", rotation=30)

plt.suptitle("BTC Performance by Fear & Greed Regime", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

print("\nRegime statistics:")
print(regime_stats.to_string())

df.drop(columns=["fng_regime"], inplace=True)

## 7. Lagged Sentiment Analysis

Does yesterday's sentiment predict today's price direction? Lag features should show stronger correlation than same-day features if sentiment leads price.

In [ ]:
lag_cols = ["fng_normalized", "fng_normalized_lag_1", "fng_normalized_lag_3",
            "fng_normalized_lag_7", "fng_sma_7", "fng_sma_14"]

lag_corr = df[lag_cols + ["target"]].corr()["target"].drop("target")

fig, ax = plt.subplots(figsize=(10, 5))
colors_bar = ["#3498db" if abs(v) == lag_corr.abs().max() else "#95a5a6" for v in lag_corr]
bars = ax.barh(lag_corr.index, lag_corr.values, color=colors_bar, edgecolor="white")
ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_xlabel("Correlation with Target (next-day direction)")
ax.set_title("Fear & Greed Lag Features vs. Target", fontsize=13, fontweight="bold")

for bar, val in zip(bars, lag_corr.values):
    ax.text(val + 0.002 if val >= 0 else val - 0.002, bar.get_y() + bar.get_height()/2,
            f"{val:.4f}", ha="left" if val >= 0 else "right", va="center", fontsize=10)

plt.tight_layout()
plt.show()

## 8. Volatility & Returns Over Time

Visualize market regimes — high-volatility periods often coincide with sentiment extremes.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

axes[0].plot(df.index, df["close"], color="#2c3e50", linewidth=1)
axes[0].plot(df.index, df["sma_50"], color="#3498db", linewidth=0.8, alpha=0.7, label="SMA 50")
axes[0].plot(df.index, df["sma_200"], color="#e74c3c", linewidth=0.8, alpha=0.7, label="SMA 200")
axes[0].set_ylabel("Price (USD)")
axes[0].set_title("BTC Price + Moving Averages")
axes[0].legend()

colors_ret = ["#2ecc71" if r >= 0 else "#e74c3c" for r in df["daily_return"]]
axes[1].bar(df.index, df["daily_return"] * 100, color=colors_ret, width=1, alpha=0.7)
axes[1].set_ylabel("Daily Return %")
axes[1].set_title("Daily Returns")

axes[2].fill_between(df.index, df["volatility_20d"] * 100, alpha=0.5, color="#9b59b6")
axes[2].set_ylabel("20d Volatility %")
axes[2].set_title("Rolling 20-Day Volatility")
axes[2].set_xlabel("Date")

plt.suptitle("BTC Market Overview", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Multicollinearity Check

Features with correlation > 0.9 are near-duplicates. We may drop some before modeling to reduce noise.

In [ ]:
threshold = 0.9
high_corr_pairs = []

for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) >= threshold:
            high_corr_pairs.append({
                "Feature A": corr.columns[i],
                "Feature B": corr.columns[j],
                "Correlation": round(corr.iloc[i, j], 4),
            })

if high_corr_pairs:
    pairs_df = pd.DataFrame(high_corr_pairs).sort_values("Correlation", key=abs, ascending=False)
    print(f"Found {len(pairs_df)} highly correlated pairs (|r| ≥ {threshold}):\n")
    print(pairs_df.to_string(index=False))
else:
    print(f"No pairs with |correlation| ≥ {threshold}")

## 10. Key Takeaways

Run the notebook end-to-end to fill in observations. After examining the plots above, note:

1. **Target balance**: ~51/49 split — no class imbalance issues.
2. **Sentiment–price relationship**: The Fear & Greed overlay shows how sentiment tracks market cycles.
3. **Lag feature value**: Compare same-day vs. lagged correlations to see if sentiment leads price.
4. **Multicollinearity**: Highly correlated feature pairs should be pruned — tree models handle it, but LSTM may not.
5. **Regime analysis**: If extreme fear days have higher next-day up probability, that's a contrarian signal the model can exploit.